### Imports & Load

In [ ]:
import pandas as pd
import numpy as np
import joblib
import random

from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.cluster import KMeans
import hdbscan

### Load Dataset

In [ ]:
df = pd.read_csv("Tofinal/final_personal_finance_dataset_v2.csv")

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date", "Amount"]).reset_index(drop=True)

# Load feature pipeline
tfidf = joblib.load("models/tfidf.pkl")
svd = joblib.load("models/svd.pkl")
sbert = joblib.load("models/sbert.pkl")
kmeans = joblib.load("models/kmeans.pkl")

# Rebuild clean text
df["raw_text"] = (
    df["Merchant"].fillna("").astype(str) + " " +
    df["Transaction Description"].fillna("").astype(str)
)

df["clean_desc"] = (
    df["raw_text"]
    .str.lower()
    .str.replace(r"[^a-z0-9 ]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"\b\d+\b", "<NUM>", regex=True)
    .str.strip()
)

print("[INFO] Data ready:", df.shape)

### Reconstruct Feature Space

In [ ]:
X_tfidf = tfidf.transform(df["clean_desc"])
X_tfidf_reduced = svd.transform(X_tfidf)

X_sbert = sbert.encode(
    df["clean_desc"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

X = np.hstack([X_tfidf_reduced, X_sbert])
print("[INFO] Feature matrix:", X.shape)

### Internal Cluster Validation

In [ ]:
labels = kmeans.predict(X)

sil = silhouette_score(X, labels)
dbi = davies_bouldin_score(X, labels)

print("Baseline Validation Metrics")
print(f"Silhouette Score: {sil:.4f}")
print(f"Davies-Bouldin Index: {dbi:.4f}")

### Stability Test ① — Random Subsampling

In [ ]:
def subsample_test(df, X, sample_ratio=0.7, runs=3):
    scores = []
    
    for i in range(runs):
        idx = np.random.choice(len(df), int(len(df)*sample_ratio), replace=False)
        X_sub = X[idx]

        km = KMeans(
            n_clusters=kmeans.n_clusters,
            random_state=42+i,
            n_init=10
        )
        labels_sub = km.fit_predict(X_sub)

        sil = silhouette_score(X_sub, labels_sub)
        scores.append(sil)

        print(f"Run {i+1}: Silhouette = {sil:.4f}")

    return scores

print("\n[STABILITY] Subsampling Test")
sub_scores = subsample_test(df, X)
print("Mean silhouette:", np.mean(sub_scores))

### Stability Test ② — Merchant Removal Test

In [ ]:
df_no_merchant = df.copy()
df_no_merchant["clean_desc_nom"] = (
    df_no_merchant["Transaction Description"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"[^a-z0-9 ]+", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
)

X_tfidf_nm = tfidf.transform(df_no_merchant["clean_desc_nom"])
X_tfidf_nm_red = svd.transform(X_tfidf_nm)

X_sbert_nm = sbert.encode(
    df_no_merchant["clean_desc_nom"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True
)

X_nm = np.hstack([X_tfidf_nm_red, X_sbert_nm])

labels_nm = kmeans.predict(X_nm)

sil_nm = silhouette_score(X_nm, labels_nm)

print("\nMerchant Removal Test")
print(f"Silhouette (no merchant): {sil_nm:.4f}")
print(f"Original silhouette     : {sil:.4f}")

### Algorithm Comparison — KMeans vs HDBSCAN

In [ ]:
hdb = hdbscan.HDBSCAN(
    min_cluster_size=100,
    metric="euclidean"
)

hdb_labels = hdb.fit_predict(X)

noise_ratio = (hdb_labels == -1).mean()
valid_labels = hdb_labels[hdb_labels != -1]

if len(set(valid_labels)) > 1:
    sil_hdb = silhouette_score(X[hdb_labels != -1], valid_labels)
else:
    sil_hdb = -1

print("\nAlgorithm Comparison")
print(f"KMeans clusters  : {kmeans.n_clusters}")
print(f"HDBSCAN clusters : {len(set(valid_labels))}")
print(f"HDBSCAN noise %  : {noise_ratio*100:.2f}%")
print(f"HDBSCAN silhouette (excl noise): {sil_hdb:.4f}")

### Per-User Cluster Distribution

In [ ]:
user_cluster_dist = (
    df.assign(ClusterID=labels)
      .groupby(["UserID", "ClusterID"])
      .size()
      .reset_index(name="Count")
)

user_cluster_dist.head()

In [ ]:
top_clusters = (
    user_cluster_dist
    .sort_values(["UserID", "Count"], ascending=False)
    .groupby("UserID")
    .head(5)
)

top_clusters

### Validation Summary

In [ ]:
validation_summary = {
    "Model": "Hybrid TF-IDF + SBERT + KMeans",
    "Clusters": kmeans.n_clusters,
    "Silhouette": round(sil, 4),
    "DBI": round(dbi, 4),
    "Merchant Removal Stable": sil_nm > sil * 0.7,
    "Chosen Model": "KMeans"
}

pd.DataFrame([validation_summary])